# 🗂️ LAB D11 – Gestión de Metadatos
## Fundamentos de Gestión de Datos | TECSUP 2026-I
**Docente:** Mg. Pilar Rocío Sayán Mejía  
**Semana:** 11 | **Unidad:** III – Gestión y Calidad del Dato


## 🎯 Capacidad a Lograr
Implementar un catálogo de metadatos con estándares Dublin Core e ISO 11179, aplicando trazabilidad de datos en un sistema de base de datos SQLite.


---
## ⚙️ Configuración Inicial – Conexión a la Base de Datos


In [ ]:
# Importación de librerías necesarias
import sqlite3
import pandas as pd
import os
import json
from datetime import datetime

print("✅ Librerías importadas correctamente")
print(f"🐍 Pandas versión: {pd.__version__}")


In [ ]:
# ─── CONEXIÓN A TU BASE DE DATOS DEL PROYECTO ───
# Opción A: Tu proyecto real
# DB_PATH = "/content/drive/MyDrive/FGD/mi_proyecto.db"

# Opción B: Base de datos de ejemplo (para ejecutar sin proyecto real)
DB_PATH = ":memory:"

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Crear tablas de ejemplo si usamos memoria
cursor.executescript("""
CREATE TABLE IF NOT EXISTS clientes (
    id_cliente    INTEGER PRIMARY KEY,
    nombre        TEXT NOT NULL,
    email         TEXT UNIQUE,
    telefono      TEXT,
    ciudad        TEXT,
    fecha_registro TEXT DEFAULT CURRENT_DATE
);

CREATE TABLE IF NOT EXISTS productos (
    id_producto   INTEGER PRIMARY KEY,
    nombre        TEXT NOT NULL,
    categoria     TEXT,
    precio        REAL CHECK(precio > 0),
    stock         INTEGER DEFAULT 0,
    proveedor     TEXT
);

CREATE TABLE IF NOT EXISTS ventas (
    id_venta      INTEGER PRIMARY KEY,
    id_cliente    INTEGER REFERENCES clientes(id_cliente),
    id_producto   INTEGER REFERENCES productos(id_producto),
    cantidad      INTEGER,
    fecha_venta   TEXT DEFAULT CURRENT_DATE,
    monto_total   REAL
);
""")
conn.commit()

# Insertar datos de ejemplo
cursor.executescript("""
INSERT OR IGNORE INTO clientes VALUES (1,'Ana Torres','ana@mail.com','999111222','Lima','2025-01-10');
INSERT OR IGNORE INTO clientes VALUES (2,'Luis Ríos','luis@mail.com','988333444','Arequipa','2025-02-05');
INSERT OR IGNORE INTO clientes VALUES (3,'María Paz','maria@mail.com','977555666','Cusco','2025-03-12');
INSERT OR IGNORE INTO productos VALUES (1,'Laptop','Tecnología',3200.0,15,'TechPeru SAC');
INSERT OR IGNORE INTO productos VALUES (2,'Mouse','Accesorios',45.0,100,'PerifericosSA');
INSERT OR IGNORE INTO productos VALUES (3,'Teclado','Accesorios',120.0,60,'PerifericosSA');
INSERT OR IGNORE INTO ventas VALUES (1,1,1,1,CURRENT_DATE,3200.0);
INSERT OR IGNORE INTO ventas VALUES (2,2,2,3,CURRENT_DATE,135.0);
INSERT OR IGNORE INTO ventas VALUES (3,3,3,2,CURRENT_DATE,240.0);
""")
conn.commit()

print("✅ Conexión establecida y datos cargados")


---
## 📋 Actividad 1 – Marco Conceptual de Metadatos

Completa la siguiente tabla con tus propias palabras basándote en los conceptos revisados en clase:

| N° | Concepto | Definición | Ejemplo aplicado |
|----|----------|------------|-----------------|
| 1 | Metadato | | |
| 2 | Dublin Core | | |
| 3 | ISO 11179 | | |
| 4 | Catálogo de datos | | |
| 5 | Trazabilidad | | |
| 6 | Data Lineage | | |
| 7 | Esquema de metadatos | | |
| 8 | Gobernanza de metadatos | | |

> 💡 **Tip:** Un metadato es "dato sobre el dato". Piensa cómo cada estándar organiza esa información.


---
## 🛠️ Actividad 2 – Implementación del Catálogo de Metadatos

### Paso 1: Explorar la Estructura de la Base de Datos


In [ ]:
# PASO 1: Explorar tablas y columnas disponibles
print("=" * 60)
print("📊 EXPLORACIÓN DE LA BASE DE DATOS")
print("=" * 60)

# Obtener lista de tablas
cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
tablas = [row[0] for row in cursor.fetchall()]
print(f"\n🗄️  Tablas encontradas: {tablas}")

# Para cada tabla, mostrar estructura
for tabla in tablas:
    print(f"\n{'─'*50}")
    print(f"📋 Tabla: {tabla.upper()}")
    cursor.execute(f"PRAGMA table_info({tabla})")
    cols = cursor.fetchall()
    df_cols = pd.DataFrame(cols, columns=['cid','name','type','notnull','default_value','pk'])
    print(df_cols[['name','type','notnull','pk']].to_string(index=False))
    cursor.execute(f"SELECT COUNT(*) FROM {tabla}")
    n = cursor.fetchone()[0]
    print(f"   → Registros: {n}")


### Paso 2: Construir el Catálogo de Metadatos (ISO 11179 + Dublin Core)


In [ ]:
# PASO 2: Construir catálogo de metadatos completo
print("=" * 60)
print("📖 CATÁLOGO DE METADATOS – ESTÁNDAR ISO 11179")
print("=" * 60)

catalogo_registros = []
hoy = datetime.now().strftime("%Y-%m-%d")

for tabla in tablas:
    cursor.execute(f"PRAGMA table_info({tabla})")
    columnas = cursor.fetchall()
    
    # Estadísticas de la tabla
    df_tabla = pd.read_sql_query(f"SELECT * FROM {tabla}", conn)
    total_filas = len(df_tabla)
    
    for col in columnas:
        col_name = col[1]
        col_type = col[2]
        not_null  = col[3]
        es_pk     = col[5]
        
        # Calcular completitud
        if col_name in df_tabla.columns and total_filas > 0:
            nulos = df_tabla[col_name].isnull().sum()
            completitud = round((1 - nulos/total_filas) * 100, 1)
        else:
            completitud = 100.0
        
        catalogo_registros.append({
            # Identificación (ISO 11179)
            "tabla":          tabla,
            "nombre_campo":   col_name,
            "tipo_dato":      col_type if col_type else "TEXT",
            "es_clave":       "PK" if es_pk else ("NOT NULL" if not_null else "Opcional"),
            # Descripción semántica
            "descripcion":    f"Campo {col_name} de la entidad {tabla}",
            "dominio":        _inferir_dominio(col_name, col_type),
            # Dublin Core
            "dc_identifier":  f"{tabla}.{col_name}",
            "dc_type":        _tipo_dublin_core(col_type),
            "dc_source":      "Base de datos del proyecto FGD",
            "dc_date":        hoy,
            "dc_rights":      "Uso académico – TECSUP 2026",
            # Calidad
            "completitud_pct": completitud,
            "total_registros": total_filas,
            # Trazabilidad
            "origen":         "Carga inicial del proyecto",
            "ultima_revision": hoy,
        })

def _inferir_dominio(nombre, tipo):
    nombre_l = nombre.lower()
    if 'precio' in nombre_l or 'monto' in nombre_l: return 'Numérico positivo'
    if 'fecha' in nombre_l or 'date' in nombre_l:   return 'Fecha ISO 8601'
    if 'email' in nombre_l:                          return 'Correo electrónico'
    if 'id' in nombre_l:                             return 'Entero autoincremental'
    if tipo and 'INT' in tipo.upper():               return 'Entero'
    if tipo and 'REAL' in tipo.upper():              return 'Decimal'
    return 'Texto libre'

def _tipo_dublin_core(tipo):
    if not tipo: return 'Text'
    tipo_u = tipo.upper()
    if 'INT' in tipo_u:  return 'Integer'
    if 'REAL' in tipo_u or 'FLOAT' in tipo_u: return 'Decimal'
    if 'DATE' in tipo_u: return 'Date'
    return 'Text'

# Reconstruir con funciones definidas
catalogo_registros = []
for tabla in tablas:
    cursor.execute(f"PRAGMA table_info({tabla})")
    columnas = cursor.fetchall()
    df_tabla = pd.read_sql_query(f"SELECT * FROM {tabla}", conn)
    total_filas = len(df_tabla)
    for col in columnas:
        col_name = col[1]; col_type = col[2]; not_null = col[3]; es_pk = col[5]
        if col_name in df_tabla.columns and total_filas > 0:
            nulos = df_tabla[col_name].isnull().sum()
            completitud = round((1 - nulos/total_filas)*100, 1)
        else:
            completitud = 100.0
        catalogo_registros.append({
            "tabla": tabla, "nombre_campo": col_name,
            "tipo_dato": col_type if col_type else "TEXT",
            "es_clave": "PK" if es_pk else ("NOT NULL" if not_null else "Opcional"),
            "descripcion": f"Campo {col_name} de la entidad {tabla}",
            "dominio": _inferir_dominio(col_name, col_type),
            "dc_identifier": f"{tabla}.{col_name}",
            "dc_type": _tipo_dublin_core(col_type),
            "dc_source": "Base de datos del proyecto FGD",
            "dc_date": hoy, "dc_rights": "Uso académico – TECSUP 2026",
            "completitud_pct": completitud, "total_registros": total_filas,
            "origen": "Carga inicial del proyecto", "ultima_revision": hoy,
        })

df_catalogo = pd.DataFrame(catalogo_registros)
print(f"\n✅ Catálogo generado: {len(df_catalogo)} campos documentados")
print(f"   Tablas cubiertas : {df_catalogo['tabla'].nunique()}")
print("\n📌 Vista previa:")
print(df_catalogo[['tabla','nombre_campo','tipo_dato','es_clave','dominio','completitud_pct']].to_string(index=False))


### Paso 3: Análisis por Tipo de Dato


In [ ]:
# PASO 3: Análisis estadístico por tipo de dato
print("=" * 60)
print("📊 ANÁLISIS POR TIPO DE DATO")
print("=" * 60)

resumen = df_catalogo.groupby(['tabla','dc_type']).agg(
    campos=('nombre_campo','count'),
    completitud_promedio=('completitud_pct','mean')
).reset_index()

print("\n📋 Distribución de tipos por tabla:")
print(resumen.to_string(index=False))

print("\n📊 Completitud promedio global por tabla:")
comp_tabla = df_catalogo.groupby('tabla')['completitud_pct'].mean().round(1)
for tabla, pct in comp_tabla.items():
    barra = "█" * int(pct/5) + "░" * (20-int(pct/5))
    print(f"  {tabla:15s} [{barra}] {pct}%")


### Paso 4: Exportar Catálogo a CSV y Ficha Técnica


In [ ]:
# PASO 4: Exportar catálogo de metadatos
import io

# Exportar a CSV en memoria (compatible con Colab)
csv_buffer = io.StringIO()
df_catalogo.to_csv(csv_buffer, index=False, encoding='utf-8')
csv_content = csv_buffer.getvalue()

# Mostrar primeras líneas del CSV
print("📄 Vista previa del CSV exportado:")
for i, linea in enumerate(csv_content.split('\n')[:5]):
    print(f"  {linea}")
print(f"  ... ({len(df_catalogo)} filas en total)")

# Guardar en Colab si hay sistema de archivos
try:
    with open('/content/catalogo_metadatos.csv', 'w', encoding='utf-8') as f:
        f.write(csv_content)
    print("\n✅ Archivo guardado: /content/catalogo_metadatos.csv")
except:
    print("\n💡 Para guardar el CSV usa: df_catalogo.to_csv('catalogo_metadatos.csv', index=False)")

# Ficha técnica JSON
ficha_tecnica = {
    "proyecto": "Fundamentos de Gestión de Datos – FGD",
    "version_catalogo": "1.0",
    "fecha_creacion": hoy,
    "autor": "Pilar Rocío Sayán Mejía",
    "estandares_aplicados": ["ISO 11179", "Dublin Core 1.1"],
    "tablas_documentadas": tablas,
    "total_campos": len(df_catalogo),
    "completitud_global": round(df_catalogo['completitud_pct'].mean(), 1)
}
print("\n📋 FICHA TÉCNICA DEL CATÁLOGO:")
print(json.dumps(ficha_tecnica, indent=2, ensure_ascii=False))


### Paso 5: Trazabilidad y Linaje de Datos


In [ ]:
# PASO 5: Registro de linaje de datos (Data Lineage)
print("=" * 60)
print("🔗 TRAZABILIDAD Y LINAJE DE DATOS")
print("=" * 60)

# Crear tabla de linaje en SQLite
cursor.execute("""
CREATE TABLE IF NOT EXISTS lineaje_datos (
    id_evento    INTEGER PRIMARY KEY AUTOINCREMENT,
    tabla_origen TEXT,
    campo        TEXT,
    operacion    TEXT,  -- INSERT, UPDATE, DELETE, SELECT
    usuario      TEXT,
    fecha_hora   TEXT,
    descripcion  TEXT
)
""")

# Registrar eventos de linaje
eventos_linaje = [
    ("clientes",  "todos",      "INSERT", "sistema",  "Carga inicial de clientes desde CSV de ventas"),
    ("productos", "todos",      "INSERT", "sistema",  "Carga inicial de catálogo de productos"),
    ("ventas",    "todos",      "INSERT", "sistema",  "Registro de transacciones del período"),
    ("clientes",  "email",      "UPDATE", "admin",    "Normalización de correos electrónicos"),
    ("productos", "precio",     "UPDATE", "admin",    "Actualización trimestral de precios"),
    ("ventas",    "monto_total","SELECT", "analista", "Generación de reporte mensual de ventas"),
]

for evento in eventos_linaje:
    cursor.execute("""
        INSERT INTO lineaje_datos (tabla_origen, campo, operacion, usuario, fecha_hora, descripcion)
        VALUES (?, ?, ?, ?, datetime('now'), ?)
    """, evento)
conn.commit()

# Consultar linaje
df_linaje = pd.read_sql_query(
    "SELECT tabla_origen, campo, operacion, usuario, fecha_hora, descripcion FROM lineaje_datos",
    conn
)
print("\n📜 Registro de linaje de datos:")
print(df_linaje.to_string(index=False))

print("\n📊 Operaciones por tabla:")
print(df_linaje.groupby(['tabla_origen','operacion']).size().reset_index(name='count').to_string(index=False))


### Paso 6: Guardar Catálogo en SQLite


In [ ]:
# PASO 6: Persistir catálogo en la base de datos
print("=" * 60)
print("💾 PERSISTENCIA DEL CATÁLOGO EN SQLITE")
print("=" * 60)

# Guardar catálogo como tabla en SQLite
df_catalogo.to_sql('catalogo_metadatos', conn, if_exists='replace', index=False)
conn.commit()

# Verificar
cursor.execute("SELECT COUNT(*) FROM catalogo_metadatos")
n = cursor.fetchone()[0]
print(f"\n✅ Tabla 'catalogo_metadatos' creada con {n} registros")

# Consulta de ejemplo sobre el catálogo
print("\n🔍 Campos clave (PK) en la base de datos:")
df_pks = pd.read_sql_query(
    "SELECT tabla, nombre_campo, tipo_dato, dominio FROM catalogo_metadatos WHERE es_clave='PK'",
    conn
)
print(df_pks.to_string(index=False))

print("\n🔍 Campos con completitud < 100%:")
df_inc = pd.read_sql_query(
    "SELECT tabla, nombre_campo, completitud_pct FROM catalogo_metadatos WHERE completitud_pct < 100 ORDER BY completitud_pct",
    conn
)
if len(df_inc) > 0:
    print(df_inc.to_string(index=False))
else:
    print("   ✅ Todos los campos tienen completitud del 100%")

conn.close()
print("\n🎉 ¡Catálogo de metadatos completado exitosamente!")


---
## 💬 Actividad 3 – Análisis Reflexivo

Responde las siguientes preguntas con base en lo desarrollado en esta sesión:

**A) ¿Cuál es la diferencia entre un metadato estructural y un metadato descriptivo? Da un ejemplo de cada uno con tus tablas.**

> _Tu respuesta aquí_

**B) ¿Por qué es importante el linaje de datos en un sistema empresarial? ¿Qué problemas puede prevenir?**

> _Tu respuesta aquí_

**C) Compara el estándar Dublin Core con ISO 11179. ¿En qué contextos aplicarías cada uno?**

> _Tu respuesta aquí_

**D) Si tu base de datos tuviera 50 tablas con 20 columnas cada una, ¿cómo automatizarías la generación del catálogo? ¿Qué información adicional agregarías?**

> _Tu respuesta aquí_


---
## ✅ Conclusiones

1. _Escribe aquí tu primera conclusión sobre la gestión de metadatos_
2. _Escribe aquí tu segunda conclusión sobre los estándares Dublin Core / ISO 11179_
3. _Escribe aquí tu tercera conclusión sobre la trazabilidad y el linaje de datos_


---
## 📚 Material Complementario

| Recurso | Enlace |
|---------|--------|
| Dublin Core Metadata Initiative | https://dublincore.org |
| ISO 11179 – Metadata Registry | https://www.iso.org/standard/61451.html |
| Data Catalog con Python | https://docs.python.org/3/library/sqlite3.html |
| DAMA-DMBOK – Gestión de Metadatos | https://www.dama.org/cpages/body-of-knowledge |
| SQLite PRAGMA table_info | https://www.sqlite.org/pragma.html#pragma_table_info |
